In [11]:
import sys
import os
import re
import importlib

import torch
import pandas as pd

from data_pipeline import (
    deduplicate,
    SimpleTokenizer,
    tokenize_corpus,
    pack_sequences,
    PreTrainingDataLoader,
    clean_text
)
import mini_gpt_torch
importlib.reload(mini_gpt_torch)
from mini_gpt_torch import MiniGPT, cross_entropy_loss, generate

print("Import Packages from mini_gpt_torch and data_pipline")





Import Packages from mini_gpt_torch and data_pipline


In [15]:
sampel_1 = "i am mohsen mohebbi"
clean_text(sampel_1)
sampel_2 = ("سلام Mohsen! این یک متن تستی است.\nHello World 123\nقیمت محصول: 2500 تومان\nEmail: test@example.com\nPython و Machine Learning خیلی جالب هستند.")
clean_text(sampel_2)

'Mohsen! . Hello World 123 : 2500 Email: test@example.com Python Machine Learning .'

1. def clear_data is NOT for farsi

In [16]:
def clean_persian_text(text: str) -> str:
    """Persian clear text 
    """
    if not isinstance(text, str):
        return ""
    # remove html
    text = re.sub(r"<[^>]+>", " ", text)
    # remove link
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # standart space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


Load dataset


In [ ]:
print("1. Load dataset".center(60, "-"))
df = pd.read_csv("comments_raw_canonical.csv")
df = df[["specialty", "text"]]
df.dropna(subset=["text"], inplace=True)

raw_docs = []
for _, row in df.iterrows():
    cleaned = clean_persian_text(str(row["text"]))
    if len(cleaned.split()) >= 3:  # comments more than  3 vocabes
        spec = clean_persian_text(str(row["specialty"])) if pd.notna(row["specialty"]) else ""
        doc = f"[{spec}] {cleaned}" if spec else cleaned
        raw_docs.append(doc)

print(f"Count comment is -----  {len(raw_docs):,}")

------------------------Load dataset------------------------
Count comment is -----  45,454


MinHash/LSH
remove dublicate doc

In [ ]:
print("2. MinHash/LSH".center(60, "-"))
sample_size = min(15000, len(raw_docs))
docs_sample = raw_docs[:sample_size]
print(f"\n remove dublicate {sample_size:,} doc")
unique_docs, removed_count = deduplicate(docs_sample, threshold=0.85, num_hashes=64, bands=8)
print(f"count doc unique : {len(unique_docs):,} (count removed comment {removed_count:,})")


------------------------MinHash/LSH-------------------------

 remove dublicate 15,000 doc
count doc unique : 14,620 (count removed comment 380)


Tokenizetr BPE

In [44]:
print("3. Learn SimpleTokenizer".center(60, "-"))

corpus_sample_for_bpe = "\n".join(unique_docs[:3000])
tokenizer = SimpleTokenizer(vocab_size=256)
tokenizer.train_bpe(corpus_sample_for_bpe, num_merges=300)

vocab_size = tokenizer.vocab_size()
print(f"end Vocab Size: {vocab_size}")


------------------3. Learn SimpleTokenizer------------------
end Vocab Size: 557


In [45]:
print("4. change doc to tokens".center(60, "-"))
flat_tokens = tokenize_corpus(unique_docs, tokenizer)

SEQ_LEN = 64
BATCH_SIZE = 16
packed_seqs, att_masks = pack_sequences(flat_tokens, seq_length=SEQ_LEN + 1, pad_id=tokenizer.pad_id)
print(f"Number of sequences ready for training {len(packed_seqs):,}")

train_loader = PreTrainingDataLoader(packed_seqs, att_masks, batch_size=BATCH_SIZE, shuffle=True)


------------------4. change doc to tokens-------------------
Number of sequences ready for training 8,038


create model

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("5. Create model".center(60, "-"))
print(f"Use divece:------- {device}")

model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=SEQ_LEN,
    ff_dim=512
).to(device)

print(f"Count parameters model {model.count_parameters():,}")


----------------------5. Create model-----------------------
Use divece:------- cuda
Count parameters model 870,784


leraning loop

In [47]:
print("6. start learning".center(60, "-"))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
EPOCHS = 2
MAX_STEPS = 1000  # speed test
LOG_EVERY = 50

model.train()
step = 0
stopped = False

for epoch in range(1, EPOCHS + 1):
    for batch_x, _ in train_loader:
        step += 1

        # change to tensor
        batch_tensor = torch.tensor(batch_x, dtype=torch.long, device=device)

        x = batch_tensor[:, :-1].contiguous()
        y = batch_tensor[:, 1:].contiguous()

        logits = model(x)
        loss = cross_entropy_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if step == 1 or step % LOG_EVERY == 0:
            print(f"Epoch {epoch} | Step {step:4d} | Loss: {loss.item():.4f}")

        if step >= MAX_STEPS:
            stopped = True
            break
    if stopped:
        break

print(f"\Training completed Final loss---- {loss.item():.4f}")


<>:38: SyntaxWarning: invalid escape sequence '\T'
<>:38: SyntaxWarning: invalid escape sequence '\T'
C:\Users\mohse\AppData\Local\Temp\ipykernel_26156\2384513908.py:38: SyntaxWarning: invalid escape sequence '\T'
  print(f"\Training completed Final loss---- {loss.item():.4f}")


---------------------6. start learning----------------------
Epoch 1 | Step    1 | Loss: 6.3441
Epoch 1 | Step   50 | Loss: 4.9009
Epoch 1 | Step  100 | Loss: 4.2284
Epoch 1 | Step  150 | Loss: 3.9170
Epoch 1 | Step  200 | Loss: 3.6756
Epoch 1 | Step  250 | Loss: 3.4090
Epoch 1 | Step  300 | Loss: 3.1153
Epoch 1 | Step  350 | Loss: 2.9721
Epoch 1 | Step  400 | Loss: 3.1505
Epoch 1 | Step  450 | Loss: 3.0889
Epoch 1 | Step  500 | Loss: 2.7909
Epoch 2 | Step  550 | Loss: 2.7282
Epoch 2 | Step  600 | Loss: 2.6889
Epoch 2 | Step  650 | Loss: 2.7119
Epoch 2 | Step  700 | Loss: 2.3448
Epoch 2 | Step  750 | Loss: 2.6247
Epoch 2 | Step  800 | Loss: 2.2810
Epoch 2 | Step  850 | Loss: 2.3359
Epoch 2 | Step  900 | Loss: 2.3034
Epoch 2 | Step  950 | Loss: 2.4839
Epoch 2 | Step 1000 | Loss: 1.9138
\Training completed Final loss---- 1.9138


Change to tokens

In [ ]:

# ============================================================
# 7. حلقه آموزش
# ============================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
EPOCHS = 2
MAX_STEPS = 1000  # حد استپ برای تست سریع
LOG_EVERY = 50

model.train()
print("\nشروع آموزش مدل...")
step = 0
stopped = False

for epoch in range(1, EPOCHS + 1):
    for batch_x, _ in train_loader:
        step += 1

        # تبدیل به تنسور (long برای embedding lookup)
        batch_tensor = torch.tensor(batch_x, dtype=torch.long, device=device)

        # جداسازی ورودی و تارگت
        # نکته: اسلایس باعث non-contiguous شدن می‌شود؛
        # با contiguous() از خطای view در cross_entropy_loss جلوگیری می‌کنیم
        x = batch_tensor[:, :-1].contiguous()
        y = batch_tensor[:, 1:].contiguous()

        logits = model(x)
        loss = cross_entropy_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # پایداری آموزش
        optimizer.step()

        if step == 1 or step % LOG_EVERY == 0:
            print(f"Epoch {epoch} | Step {step:4d} | Loss: {loss.item():.4f}")

        if step >= MAX_STEPS:
            stopped = True
            break
    if stopped:
        break

print(f"\nآموزش پایان یافت. Loss نهایی: {loss.item():.4f}")

# ============================================================
# 8. تولید متن (Inference)
# ============================================================
model.eval()

test_prompt = "دکتر بسیار با حوصله"
prompt_tokens = tokenizer.encode(test_prompt)
print(f"\nPrompt: '{test_prompt}'")
print("در حال تولید ادامه نظر...")

output_tokens = generate(model, prompt_tokens, max_new_tokens=40, temperature=0.7)
generated_text = tokenizer.decode(output_tokens)
print(f"\nGenerated Result:\n{generated_text}")


در حال بارگذاری داده‌های کامنت‌ها...
تعداد کل نظرات فیلترشده اولیه: 45,454

در حال حذف تکراری‌ها روی 15,000 سند...
تعداد اسناد منحصر‌به‌فرد: 14,620 (تعداد تکراری حذف‌شده: 380)

در حال آموزش SimpleTokenizer (BPE)...
اندازه واژگان نهایی (Vocab Size): 557

در حال تبدیل کل اسناد به توکن...
تعداد توالی‌های آماده برای آموزش: 8,038

دستگاه پردازش: cuda
تعداد کل پارامترهای مدل: 870,784

شروع آموزش مدل...
Epoch 1 | Step    1 | Loss: 6.3623
Epoch 1 | Step   50 | Loss: 4.9192
Epoch 1 | Step  100 | Loss: 4.1432
Epoch 1 | Step  150 | Loss: 3.7814
Epoch 1 | Step  200 | Loss: 3.5737
Epoch 1 | Step  250 | Loss: 3.2569
Epoch 1 | Step  300 | Loss: 3.0341
Epoch 1 | Step  350 | Loss: 3.3520
Epoch 1 | Step  400 | Loss: 3.0886
Epoch 1 | Step  450 | Loss: 2.8941
Epoch 1 | Step  500 | Loss: 2.8617
Epoch 2 | Step  550 | Loss: 2.4480
Epoch 2 | Step  600 | Loss: 2.5277
Epoch 2 | Step  650 | Loss: 2.8625
Epoch 2 | Step  700 | Loss: 2.5656
Epoch 2 | Step  750 | Loss: 2.2534
Epoch 2 | Step  800 | Loss: 2.5774
Epoc